# Arize AX Get Started Tutorial

This notebook accompanies the [Arize AX Get Started guide](https://arize.com/docs/ax/get-started). It builds a simple airline customer-service RAG chatbot called **SkyServe** and walks you through the full AX development workflow:

1. **Tracing** - Instrument your app and send traces to Arize AX
2. **Evaluations** - Automatically measure response quality
3. **Prompt Playground & Prompt Hub** - Iterate on prompts using real data
4. **Datasets & Experiments** - Prove your changes work
5. **Dashboards & Monitors** - Stay on top of production

## Prerequisites

- An [Arize AX account](https://app.arize.com/auth/join) (free)
- An [OpenAI API key](https://platform.openai.com/api-keys)
- Python 3.9+

## Setup

Install the required packages:

In [ ]:
!pip install arize-otel openai openinference-instrumentation-openai chromadb

Set your API keys. You can find your Arize Space ID and API Key on the **Settings** page in Arize AX.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

ARIZE_SPACE_ID = getpass("Arize Space ID: ")
ARIZE_API_KEY = getpass("Arize API Key: ")

---
# Part 1: Tracing

In this section we build the SkyServe chatbot and instrument it so every request is traced in Arize AX.

## 1.1 Create the airline policy documents

These are the knowledge-base documents that our RAG chatbot will retrieve from.

In [ ]:
AIRLINE_POLICIES = [
    {
        "title": "Refund Policy",
        "content": (
            "SkyServe Airlines Refund Policy:\n"
            "- Full refund: Tickets canceled within 24 hours of purchase, regardless of fare type.\n"
            "- Refundable tickets: Can be canceled anytime for a full refund minus a $25 processing fee.\n"
            "- Non-refundable tickets: No cash refund. A travel credit is issued minus a $75 change fee. "
            "Credits expire 12 months from the original purchase date.\n"
            "- Flights canceled by SkyServe: Passengers receive a full refund automatically within 7 business days.\n"
            "- Refund requests must be submitted through the website or by calling customer support."
        ),
    },
    {
        "title": "Baggage Allowance",
        "content": (
            "SkyServe Airlines Baggage Policy:\n"
            "- Personal item (e.g., purse, laptop bag): Free on all fares. Must fit under the seat.\n"
            "- Carry-on bag: Free on Plus and Premium fares. $35 per bag on Basic fares. "
            "Max dimensions: 22 x 14 x 9 inches.\n"
            "- First checked bag: Free on Premium fares. $30 on Plus fares. $40 on Basic fares.\n"
            "- Second checked bag: $45 on all fare types.\n"
            "- Overweight bags (51-70 lbs): Additional $50 surcharge.\n"
            "- Oversized bags (63-80 linear inches): Additional $75 surcharge.\n"
            "- Bags over 70 lbs or 80 linear inches are not accepted."
        ),
    },
    {
        "title": "Rebooking and Change Policy",
        "content": (
            "SkyServe Airlines Rebooking Policy:\n"
            "- Premium and Plus fares: Free unlimited changes up to 1 hour before departure.\n"
            "- Basic fares: $75 change fee per change, plus any fare difference.\n"
            "- Same-day standby: Available for Plus and Premium passengers at no charge. "
            "Basic fare passengers pay $50.\n"
            "- No-shows: Basic fare tickets are forfeited. Plus tickets receive travel credit minus $75. "
            "Premium tickets receive travel credit with no penalty.\n"
            "- Schedule changes by SkyServe of more than 2 hours: Free rebooking or full refund offered."
        ),
    },
    {
        "title": "Loyalty Program - SkyMiles",
        "content": (
            "SkyServe SkyMiles Loyalty Program:\n"
            "- Earn 1 SkyMile per dollar spent on Basic fares, 2 on Plus, 3 on Premium.\n"
            "- Tier levels: Silver (10,000 miles/year), Gold (25,000), Platinum (50,000), Diamond (100,000).\n"
            "- Silver benefits: Priority boarding, free carry-on on Basic fares.\n"
            "- Gold benefits: Silver perks + free first checked bag, lounge access 2x/year.\n"
            "- Platinum benefits: Gold perks + free changes on all fares, unlimited lounge access.\n"
            "- Diamond benefits: Platinum perks + complimentary upgrades when available, dedicated phone line.\n"
            "- Miles expire after 24 months of account inactivity."
        ),
    },
    {
        "title": "Flight Delay and Cancellation Compensation",
        "content": (
            "SkyServe Airlines Delay & Cancellation Compensation:\n"
            "- Delays under 2 hours: No compensation provided.\n"
            "- Delays of 2-4 hours: $50 travel voucher for future SkyServe flights.\n"
            "- Delays over 4 hours: Rebooking on next available flight or full refund. "
            "Meal voucher ($15) provided at the airport.\n"
            "- Overnight delays: Hotel accommodation arranged by SkyServe if the delay is within airline control. "
            "Transportation to/from hotel included.\n"
            "- Cancellations: Full refund or rebooking on next available flight. "
            "If cancellation is within airline control, $100 travel voucher issued in addition.\n"
            "- Weather-related disruptions: Rebooking at no charge, but no additional compensation."
        ),
    },
]

## 1.2 Build the vector store

We use ChromaDB as a lightweight in-memory vector store. OpenAI embeddings convert our policy documents into vectors so the chatbot can find the most relevant policy for each question.

In [ ]:
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="airline_policies")

collection.add(
    documents=[doc["content"] for doc in AIRLINE_POLICIES],
    metadatas=[{"title": doc["title"]} for doc in AIRLINE_POLICIES],
    ids=[f"policy_{i}" for i in range(len(AIRLINE_POLICIES))],
)

print(f"Indexed {collection.count()} policy documents.")

## 1.3 Set up Arize AX tracing

These few lines are all it takes to send every OpenAI call to Arize AX as a trace.

In [ ]:
from arize.otel import register, Endpoint
from openinference.instrumentation.openai import OpenAIInstrumentor

tracer_provider = register(
    space_id=ARIZE_SPACE_ID,
    api_key=ARIZE_API_KEY,
    project_name="skyserve-chatbot",
    endpoint=Endpoint.ARIZE,
)

OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

print("Tracing enabled! All OpenAI calls will now appear in Arize AX.")

## 1.4 Build the RAG chatbot

Our `ask_skyserve` function:
1. Retrieves the most relevant policy document from ChromaDB
2. Sends the question + context to OpenAI
3. Returns the response

In [ ]:
import openai

oai_client = openai.OpenAI()

SYSTEM_PROMPT = """You are SkyServe Airlines' customer service assistant. 
Answer the customer's question based on the provided policy documents. 
Be friendly and helpful."""


def ask_skyserve(question: str) -> str:
    """Answer a customer question using RAG over airline policy documents."""
    # Step 1: Retrieve relevant policy
    results = collection.query(query_texts=[question], n_results=2)
    context = "\n\n".join(results["documents"][0])

    # Step 2: Generate response
    response = oai_client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Policy documents:\n{context}\n\nCustomer question: {question}",
            },
        ],
    )
    return response.choices[0].message.content

## 1.5 Generate traces

Let's run a set of customer questions through the chatbot. These include straightforward questions, edge cases, and some tricky ones where the chatbot might struggle.

In [ ]:
SAMPLE_QUESTIONS = [
    "Can I get a refund on my Basic fare ticket I bought 3 days ago?",
    "How much does a carry-on bag cost?",
    "I'm a Gold SkyMiles member. Do I get free checked bags?",
    "My flight was delayed 5 hours. What am I entitled to?",
    "Can I change my Basic fare flight for free?",
    "What happens if I don't show up for my flight?",
    "How many SkyMiles do I earn on a Premium ticket?",
    "Is there a weight limit for checked bags?",
    "My flight was canceled due to weather. Do I get a hotel?",
    "I bought a non-refundable ticket yesterday. Can I still get my money back?",
    "What are the dimensions for carry-on bags?",
    "I'm a Silver member flying Basic. Do I get a free carry-on?",
    "Can I get a refund if SkyServe cancels my flight?",
    "How do I avoid paying for a carry-on bag?",
    "When do my SkyMiles expire?",
]

print("Sending questions to SkyServe chatbot...\n")
for q in SAMPLE_QUESTIONS:
    answer = ask_skyserve(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")
    print("---\n")

print("Done! Check your Arize AX project 'skyserve-chatbot' to see the traces.")

---
# Part 2: Evaluations

Now that traces are flowing into Arize AX, set up **online evaluations** in the UI to automatically score every response. Follow the steps in the [Evaluations guide](https://arize.com/docs/ax/get-started/get-started-evaluations) to:

1. Create an LLM-as-a-Judge evaluator for **Helpfulness**
2. Create a code-based evaluator to check for **empty responses**
3. View evaluation scores on your traces

These evaluations are configured entirely in the AX UI - no code required.

---
# Part 3: Prompt Playground & Prompt Hub

With evaluation scores on your traces, you can now identify low-scoring responses and improve them. Follow the steps in the [Prompts guide](https://arize.com/docs/ax/get-started/get-started-prompts) to:

1. Replay a low-scoring trace in the Prompt Playground
2. Improve the system prompt
3. Save the improved prompt to Prompt Hub

Once you've saved an improved prompt, you can pull it into your app code:

In [ ]:
# After saving your improved prompt to Prompt Hub (see the Prompts guide),
# you can pull it into your app like this:

from arize.experimental.prompt_hub import ArizePromptClient

prompt_client = ArizePromptClient(
    space_id=ARIZE_SPACE_ID,
    api_key=ARIZE_API_KEY,
)

# Pull the latest version of your prompt
saved_prompt = prompt_client.get_prompt(name="skyserve-support")
print("Loaded prompt from Prompt Hub:")
print(saved_prompt.messages)

---
# Part 4: Datasets & Experiments

Experiments let you prove that prompt changes actually improve quality across a representative set of queries - not just the one you noticed.

Follow the steps in the [Experiments guide](https://arize.com/docs/ax/get-started/get-started-experiments) to create a dataset, run your old and new prompts as experiments, and compare results.

Below is a sample CSV you can use to create your test dataset. Save it as `skyserve_test_cases.csv` and upload it in the AX UI.

In [ ]:
import pandas as pd

test_cases = pd.DataFrame({
    "input": [
        "Can I get a refund on my non-refundable ticket?",
        "How much does a second checked bag cost?",
        "I'm a Platinum member. Can I change my Basic fare for free?",
        "My flight was delayed 3 hours. What compensation do I get?",
        "What happens if I miss my flight with a Plus ticket?",
        "How big can my carry-on be?",
        "Do Silver members get lounge access?",
        "My flight was canceled due to a mechanical issue. What do I get?",
        "Can I get same-day standby on a Basic fare?",
        "How long do travel credits last?",
    ],
    "expected_output": [
        "No cash refund, but a travel credit is issued minus a $75 change fee. Credits expire in 12 months.",
        "$45 on all fare types.",
        "Yes, Platinum members get free changes on all fares.",
        "A $50 travel voucher for future SkyServe flights.",
        "Plus ticket no-shows receive travel credit minus $75.",
        "Maximum 22 x 14 x 9 inches.",
        "No, lounge access starts at Gold tier (2 visits/year).",
        "Full refund or rebooking, plus a $100 travel voucher since it was within airline control.",
        "Yes, for a $50 fee.",
        "Travel credits expire 12 months from the original purchase date.",
    ],
})

test_cases.to_csv("skyserve_test_cases.csv", index=False)
print("Saved skyserve_test_cases.csv - upload this in the Arize AX UI.")
test_cases

---
# Part 5: Dashboards & Monitors

With your app improved and tested, set up production monitoring. Follow the steps in the [Production guide](https://arize.com/docs/ax/get-started/get-started-production) to:

1. Create a dashboard for the SkyServe project
2. Set up managed monitors for latency and errors
3. Create an eval-based monitor for quality alerts
4. Connect alerts to Slack or email

All of this is done in the AX UI.

---

## What you've accomplished

You've completed the full Arize AX development workflow:

- **Tracing**: Full visibility into every chatbot request
- **Evaluations**: Automated quality scoring on every response
- **Prompts**: Data-driven prompt iteration with version control
- **Experiments**: Proof that your changes improve quality
- **Monitoring**: Alerts that catch regressions before your users do

For deeper dives into any of these features, visit the [Arize AX documentation](https://arize.com/docs/ax).